In [9]:
!pip install transformers torch torchaudio librosa accelerate --quiet

In [10]:
import torch
import librosa
import numpy as np
from transformers import AutoProcessor, AutoModelForSpeechSeq2Seq

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

PyTorch version: 2.10.0+cu128
CUDA available: True
Using device: cuda


In [41]:
import gc
gc.collect()

36738

In [42]:
MODEL_ID = "tarteel-ai/whisper-base-ar-quran"

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForSpeechSeq2Seq.from_pretrained(MODEL_ID)
model = model.to(device)
model.eval()

print(f"Model loaded and moved to: {device}")

Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

Model loaded and moved to: cuda


In [57]:
AUDIO_FILE = "/content/محمد أيوب رحمه الله كروما قران شاشة سوداء جاهزة للتصميم سورة الأنعام التلاوة كاملة تحت في الوصف 👇 - المصحف المرتل.mp3"   # <-- change to your MP3 file path
TARGET_SR  = 16_000             # Whisper requires 16 kHz

# Load and resample
audio, sr = librosa.load(AUDIO_FILE, sr=TARGET_SR, mono=True)

duration = len(audio) / TARGET_SR
print(f"Loaded: {AUDIO_FILE}")
print(f"Sample rate: {TARGET_SR} Hz  |  Duration: {duration:.2f}s  |  Samples: {len(audio)}")

Loaded: /content/محمد أيوب رحمه الله كروما قران شاشة سوداء جاهزة للتصميم سورة الأنعام التلاوة كاملة تحت في الوصف 👇 - المصحف المرتل.mp3
Sample rate: 16000 Hz  |  Duration: 44.40s  |  Samples: 710400


In [51]:
# Prepare input features
inputs = processor(
    audio,
    sampling_rate=TARGET_SR,
    return_tensors="pt"
)
input_features = inputs.input_features.to(device)

# --- Patch generation_config for this fine-tuned model ---
# Newer transformers (>=4.36) no longer accept forced_decoder_ids in generate();
# everything must live on generation_config instead.
tokenizer = processor.tokenizer

# 1. Timestamp token IDs (missing from fine-tuned checkpoint)
no_ts_id = tokenizer.convert_tokens_to_ids("<|notimestamps|>")
model.generation_config.no_timestamps_token_id = no_ts_id
model.generation_config.begin_suppress_tokens  = [no_ts_id]

# 2. Language + task forced prefix (replaces forced_decoder_ids kwarg)
model.generation_config.forced_decoder_ids = processor.get_decoder_prompt_ids(
    language="ar", task="transcribe"
)

# 3. Stay within max_target_positions budget
forced_len = len(model.generation_config.forced_decoder_ids)
model.generation_config.max_new_tokens = model.config.max_target_positions - forced_len - 1

print(f"no_timestamps_token_id : {no_ts_id}")
print(f"forced_decoder_ids     : {model.generation_config.forced_decoder_ids}")
print(f"max_new_tokens         : {model.generation_config.max_new_tokens}")

# Generate
with torch.no_grad():
    predicted_ids = model.generate(
        input_features,
        return_timestamps=True,
    )

print(f"Generated {predicted_ids.shape[-1]} tokens")

Both `max_new_tokens` (=444) and `max_length`(=1024) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


no_timestamps_token_id : 50363
forced_decoder_ids     : [(1, 50272), (2, 50359), (3, 50363)]
max_new_tokens         : 444
Generated 246 tokens


In [52]:
import re

# Use tokenizer.decode() directly on the flat token list to avoid batch_decode
# wrapping issues with _decode_with_timestamps in this version of transformers.
token_list  = predicted_ids[0].tolist()   # flat list of ints
raw_decoded = processor.tokenizer.decode(token_list, decode_with_timestamps=True)
print("Raw decoded output:")
print(raw_decoded)
print()

Raw decoded output:
<|0.00|>إِنِّيَا رَأَيْتَهُمْ حَسِبْتَهُمْ لُؤْلُؤًا مَنْثُورًا وَإِذَا رَأَيْتَ مَعَيْتَ نَعِيمًا وَمُلْكًا كَبِيرًا وَإِذَا رَأَيْتَ ثَمَّ رَأَيْتَ نَعِيمًا وَمُلْكًا كَبِيرًا عَالِيَهُمْ قِيَابُ سُنْدُسٍ خُضْرٌ وَإِسْتَبْرَقٌ وَحُلُّوا أَسَاوِرَ مِنْ فِضْلَهُ وَسِقَاهُمْ رَبُّهُمْ شَرَابٌ



In [53]:
def parse_timestamped_output(decoded_str, total_audio_duration):
    """
    Parses Whisper's timestamp-annotated string.
    Returns a list of dicts: {'start': float, 'end': float, 'text': str}
    Adds a fallback for cases where only a starting timestamp is present.
    """
    # Find all timestamp positions in the string
    pattern = r"<\|(\d+\.\d+)\|>"
    tokens = re.split(pattern, decoded_str)
    # tokens alternates: [pre-text, ts0, text0, ts1, text1, ts2, ...]
    segments = []
    i = 0
    # Skip anything before the first timestamp token
    while i < len(tokens) and not re.fullmatch(r"\d+\.\d+", tokens[i].strip()):
        i += 1

    # Main loop for parsing paired timestamps
    while i + 2 < len(tokens):
        start = float(tokens[i])
        text  = tokens[i + 1].strip()
        end   = float(tokens[i + 2])
        if text:  # skip empty segments
            segments.append({"start": start, "end": end, "text": text})
        i += 2

    # Handle the case where the transcription started with a timestamp but ended without another one.
    # E.g., "<|0.00|> This is the full text."
    # After the loop, if there's an unprocessed timestamp and subsequent text, add it as a segment
    # extending to the end of the audio.
    if i < len(tokens) and re.fullmatch(r"\d+\.\d+", tokens[i].strip()):
        current_start_time = float(tokens[i])
        # The text following this timestamp would be at tokens[i+1] if it exists
        remaining_text_index = i + 1
        if remaining_text_index < len(tokens):
            remaining_text = tokens[remaining_text_index].strip()
            if remaining_text:
                # This segment effectively runs from its start time to the end of the audio
                segments.append({"start": current_start_time, "end": total_audio_duration, "text": remaining_text})


    return segments


segments = parse_timestamped_output(raw_decoded, duration)

print(f"{'#':<4} {'Start':>8}  {'End':>8}  Text")
print("-" * 60)
for idx, seg in enumerate(segments, 1):
    print(f"{idx:<4} {seg['start']:>7.2f}s  {seg['end']:>7.2f}s  {seg['text']}")

#       Start       End  Text
------------------------------------------------------------
1       0.00s    48.17s  إِنِّيَا رَأَيْتَهُمْ حَسِبْتَهُمْ لُؤْلُؤًا مَنْثُورًا وَإِذَا رَأَيْتَ مَعَيْتَ نَعِيمًا وَمُلْكًا كَبِيرًا وَإِذَا رَأَيْتَ ثَمَّ رَأَيْتَ نَعِيمًا وَمُلْكًا كَبِيرًا عَالِيَهُمْ قِيَابُ سُنْدُسٍ خُضْرٌ وَإِسْتَبْرَقٌ وَحُلُّوا أَسَاوِرَ مِنْ فِضْلَهُ وَسِقَاهُمْ رَبُّهُمْ شَرَابٌ


In [58]:
from transformers import pipeline, GenerationConfig

# The fine-tuned checkpoint ships with an outdated GenerationConfig that is
# missing lang_to_id / task_to_id (and no_timestamps_token_id).
# Load a complete one from the canonical base model, then patch the missing field.
gen_cfg = GenerationConfig.from_pretrained("openai/whisper-base")
gen_cfg.no_timestamps_token_id = processor.tokenizer.convert_tokens_to_ids("<|notimestamps|>")
gen_cfg.begin_suppress_tokens  = [gen_cfg.no_timestamps_token_id]
model.generation_config = gen_cfg

asr_pipe = pipeline(
    "automatic-speech-recognition",
    model=model,                              # patched model object
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    dtype=torch.float16 if device == "cuda" else torch.float32,
    device=device,
)

word_result = asr_pipe(
    AUDIO_FILE,
    return_timestamps="word",
    generate_kwargs={"language": "ar", "task": "transcribe"},
    chunk_length_s=30,
    stride_length_s=5,
)

print(f"Full transcription:\n{word_result['text']}\n")
print(f"\n{'#':<5} {'Start':>8}  {'End':>8}  Word")
print("-" * 50)
for idx, chunk in enumerate(word_result["chunks"], 1):
    start, end = chunk["timestamp"]
    end_str = f"{end:.2f}s" if end is not None else "  ----"
    print(f"{idx:<5} {start:>7.2f}s  {end_str:>8}  {chunk['text']}")


Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).
Whisper did not predict an ending timestamp, which can happen if audio is cut off in the middle of a word. Also make sure WhisperTimeStampLogitsProcessor was used during generation.


Full transcription:
قَدْ خَسِرَ الَّذِينَ كَذَّبُوا بِلِقَاءِ اللَّهِ حَتَّى إِذَا جَاءَتْهُمُ السَّاعَةُ بَغْتَةً قَالُوا يَا حَسْرَتَنَا عَلَى مَا فَرَّطْنَا فِيهَا وَهُمْ يَحْمِلُونَ أَوْزَارَهُمْ عَلَى ظُهُورِهِمْ أَلَا سَاءَ مَا يَزِرُونَ وَمَا الْحَيَاةُ الدُّنْيَا إِلَّا لَعِبٌ وَلَهْوٌ وَلَلدَّارُ الْآخِرَةُ خَيْرٌ لِلَّذِينَ يَتَّقُونَ أَفَلَا تَعْقِلُونَ


#        Start       End  Word
--------------------------------------------------
1        0.00s     0.42s  قَدْ
2        0.42s     1.02s   خَسِرَ
3        1.02s     2.04s   الَّذِينَ
4        2.04s     3.04s   كَذَّبُوا
5        3.04s     5.04s   بِلِقَاءِ
6        5.04s     5.86s   اللَّهِ
7        5.86s     6.84s   حَتَّى
8        6.84s     7.98s   إِذَا
9        7.98s     9.96s   جَاءَتْهُمُ
10       9.96s    11.10s   السَّاعَةُ
11      11.10s    12.22s   بَغْتَةً
12      12.22s    14.60s   قَالُوا
13      14.60s    15.02s   يَا
14      15.02s    16.26s   حَسْرَتَنَا
15      16.26s    16.76s   عَلَى
16      16.76s    17